In [1]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
import joblib
from pathlib import Path

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions, RunningMode

In [2]:
# Paths
NEW_VIDEO_PATH = Path("../data/raw/test_video_data/new_video4.mp4")
MODEL_PATH    = Path("../data/models/pose_landmarker_lite.task")
OUTPUT_CSV    = Path("../data/raw/test_tabular_data/test_angles_dataset2.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete.")
print(f"   Model        : {MODEL_PATH.resolve()}")
print(f"   Video source : {NEW_VIDEO_PATH.resolve()}")
print(f"   Output CSV   : {OUTPUT_CSV.resolve()}")

✅ Setup complete.
   Model        : C:\Users\USER\Desktop\AI_PROJECT_2\data\models\pose_landmarker_lite.task
   Video source : C:\Users\USER\Desktop\AI_PROJECT_2\data\raw\test_video_data\new_video4.mp4
   Output CSV   : C:\Users\USER\Desktop\AI_PROJECT_2\data\raw\test_tabular_data\test_angles_dataset2.csv


In [3]:
def calculate_angle(a, b, c):
    """
    Calculate the angle at point B, formed by points A -> B -> C.
    
    Args:
        a, b, c: Each is a (x, y) tuple representing a joint coordinate.
    
    Returns:
        Angle in degrees (0–180).
    """
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - \
              np.arctan2(a[1] - b[1], a[0] - b[0])
    
    angle = np.abs(np.degrees(radians))
    
    # Normalize to [0, 180]
    if angle > 180:
        angle = 360 - angle

    return round(angle, 2)


# Quick sanity check
test_angle = calculate_angle((0, 0), (1, 0), (1, 1))
print(f"✅ Angle function works. Test angle (expected 90°): {test_angle}°")

✅ Angle function works. Test angle (expected 90°): 90.0°


In [4]:
# MediaPipe Pose landmark indices
LANDMARKS = {
    "left_shoulder"  : 11, "right_shoulder" : 12,
    "left_elbow"     : 13, "right_elbow"    : 14,
    "left_wrist"     : 15, "right_wrist"    : 16,
    "left_hip"       : 23, "right_hip"      : 24,
    "left_knee"      : 25, "right_knee"     : 26,
    "left_ankle"     : 27, "right_ankle"    : 28,
    "left_heel"      : 29, "right_heel"     : 30,
}

def get_coords(landmarks, name):
    """Extract (x, y) from a named landmark."""
    lm = landmarks[LANDMARKS[name]]
    return (lm.x, lm.y)

def get_z(landmarks, name):
    """Extract z (depth) from a named landmark."""
    return landmarks[LANDMARKS[name]].z


def extract_angles(landmarks):
    """
    Given a list of MediaPipe pose landmarks,
    compute and return the 10 joint angles + 3 torso rotation
    features as a dict. (13 features total — matches training data.)
    """
    angles = {}

    # --- Elbow angles: shoulder -> elbow -> wrist ---
    angles["left_elbow_angle"]  = calculate_angle(
        get_coords(landmarks, "left_shoulder"),
        get_coords(landmarks, "left_elbow"),
        get_coords(landmarks, "left_wrist")
    )
    angles["right_elbow_angle"] = calculate_angle(
        get_coords(landmarks, "right_shoulder"),
        get_coords(landmarks, "right_elbow"),
        get_coords(landmarks, "right_wrist")
    )

    # --- Shoulder angles: elbow -> shoulder -> hip ---
    angles["left_shoulder_angle"]  = calculate_angle(
        get_coords(landmarks, "left_elbow"),
        get_coords(landmarks, "left_shoulder"),
        get_coords(landmarks, "left_hip")
    )
    angles["right_shoulder_angle"] = calculate_angle(
        get_coords(landmarks, "right_elbow"),
        get_coords(landmarks, "right_shoulder"),
        get_coords(landmarks, "right_hip")
    )

    # --- Hip angles: shoulder -> hip -> knee ---
    angles["left_hip_angle"]  = calculate_angle(
        get_coords(landmarks, "left_shoulder"),
        get_coords(landmarks, "left_hip"),
        get_coords(landmarks, "left_knee")
    )
    angles["right_hip_angle"] = calculate_angle(
        get_coords(landmarks, "right_shoulder"),
        get_coords(landmarks, "right_hip"),
        get_coords(landmarks, "right_knee")
    )

    # --- Knee angles: hip -> knee -> ankle ---
    angles["left_knee_angle"]  = calculate_angle(
        get_coords(landmarks, "left_hip"),
        get_coords(landmarks, "left_knee"),
        get_coords(landmarks, "left_ankle")
    )
    angles["right_knee_angle"] = calculate_angle(
        get_coords(landmarks, "right_hip"),
        get_coords(landmarks, "right_knee"),
        get_coords(landmarks, "right_ankle")
    )

    # --- Ankle angles: knee -> ankle -> heel ---
    angles["left_ankle_angle"]  = calculate_angle(
        get_coords(landmarks, "left_knee"),
        get_coords(landmarks, "left_ankle"),
        get_coords(landmarks, "left_heel")
    )
    angles["right_ankle_angle"] = calculate_angle(
        get_coords(landmarks, "right_knee"),
        get_coords(landmarks, "right_ankle"),
        get_coords(landmarks, "right_heel")
    )

    # --- Torso rotation features (z-depth) — required for russian_twist ---
    angles["shoulder_z_diff"]  = round(get_z(landmarks, "left_shoulder") - get_z(landmarks, "right_shoulder"), 4)
    angles["hip_z_diff"]       = round(get_z(landmarks, "left_hip")      - get_z(landmarks, "right_hip"),      4)
    angles["torso_rotation"]   = round(angles["shoulder_z_diff"] - angles["hip_z_diff"], 4)

    return angles


print("✅ Landmark helpers and extract_angles() ready.")
print("   Features per frame: 10 joint angles + 3 torso rotation = 13 total.")

✅ Landmark helpers and extract_angles() ready.
   Features per frame: 10 joint angles + 3 torso rotation = 13 total.


In [5]:
def process_video(video_path, options):
    """
    Process a single video file and return a list of rows,
    where each row represents one frame's angles.

    Args:
        video_path          : Path to the video file.
        options             : PoseLandmarkerOptions (initialized outside)

    Returns:
        List of dicts, one per valid frame.
    """
    rows = []
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"   ⚠️  Could not open: {video_path.name}")
        return rows

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 30  # fallback

    prev_angles = None
    frame_number = 0

    with PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Convert frame to MediaPipe Image
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

            # Timestamp in milliseconds (required for VIDEO mode)
            timestamp_ms = int((frame_number / fps) * 1000)

            # Run pose detection
            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                landmarks = result.pose_landmarks[0]  # first (and usually only) person
                angles    = extract_angles(landmarks)

                # --- Angular velocities (change in angle per second) ---
                if prev_angles is not None:
                    angular_velocities = {
                        f"{key}_velocity": round(abs(angles[key] - prev_angles[key]) * fps, 2)
                        for key in angles
                    }
                else:
                    # First valid frame: velocity is 0
                    angular_velocities = {f"{key}_velocity": 0.0 for key in angles}

                row = {
                    "frame_number"        : frame_number,
                    **angles,
                    **angular_velocities,
                }
                rows.append(row)
                prev_angles = angles

            frame_number += 1

    cap.release()
    return rows


print("✅ Video processing function ready.")
print("   Outputs per frame: frame_number + 10 angles + 10 angular velocities")

✅ Video processing function ready.
   Outputs per frame: frame_number + 10 angles + 10 angular velocities


In [6]:
def build_dataset(video_path):
    """
    Process one new unseen video and return an unlabeled DataFrame.
    """
    all_rows = []

    # Initialize PoseLandmarker options once
    base_options = mp_python.BaseOptions(model_asset_path=str(MODEL_PATH))
    options = PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )

    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    print(f"\n🎞️ Processing: {video_path.name} ...", end=" ")
    rows = process_video(video_path, options)
    all_rows.extend(rows)
    print(f"{len(rows)} frames extracted.")

    df = pd.DataFrame(all_rows)
    return df


# --- Run it ---
print("🚀 Starting dataset creation...\n")
df_raw = build_dataset(NEW_VIDEO_PATH)

print(f"\n✅ Done! Total frames collected: {len(df_raw)}")
print(f"   Shape : {df_raw.shape}")
print(f"   Columns: {list(df_raw.columns)}")

🚀 Starting dataset creation...


🎞️ Processing: new_video4.mp4 ... 62 frames extracted.

✅ Done! Total frames collected: 62
   Shape : (62, 27)
   Columns: ['frame_number', 'left_elbow_angle', 'right_elbow_angle', 'left_shoulder_angle', 'right_shoulder_angle', 'left_hip_angle', 'right_hip_angle', 'left_knee_angle', 'right_knee_angle', 'left_ankle_angle', 'right_ankle_angle', 'shoulder_z_diff', 'hip_z_diff', 'torso_rotation', 'left_elbow_angle_velocity', 'right_elbow_angle_velocity', 'left_shoulder_angle_velocity', 'right_shoulder_angle_velocity', 'left_hip_angle_velocity', 'right_hip_angle_velocity', 'left_knee_angle_velocity', 'right_knee_angle_velocity', 'left_ankle_angle_velocity', 'right_ankle_angle_velocity', 'shoulder_z_diff_velocity', 'hip_z_diff_velocity', 'torso_rotation_velocity']


In [7]:
# --- Save raw dataset ---
df_raw.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Raw dataset saved to: {OUTPUT_CSV}")
print(f"   Shape: {df_raw.shape}\n")

# --- Sanity Checks ---
angle_cols    = [col for col in df_raw.columns if "angle" in col and "velocity" not in col]
velocity_cols = [col for col in df_raw.columns if "velocity" in col]

# 1. Preview
print("=" * 60)
print("📋 First 5 rows:")
display(df_raw.head())

# 2. Null check
print("=" * 60)
null_counts = df_raw.isnull().sum()
if null_counts.sum() == 0:
    print("✅ No null values found.")
else:
    print("⚠️  Null values detected:")
    display(null_counts[null_counts > 0])

# 3. Angle range check → DROP out-of-range rows (mirrors training cleaning)
print("=" * 60)
before = len(df_raw)
valid_mask = df_raw[angle_cols].apply(lambda col: (col >= 0) & (col <= 180)).all(axis=1)
df_raw = df_raw[valid_mask].reset_index(drop=True)
dropped = before - len(df_raw)

if dropped == 0:
    print("✅ All angles within [0°, 180°]. No rows dropped.")
else:
    print(f"⚠️  Dropped {dropped} frames with out-of-range angles. Remaining: {len(df_raw)}")

# 4. Basic statistics
print("=" * 60)
print("📈 Angle columns statistics:")
display(df_raw[angle_cols].describe())

✅ Raw dataset saved to: ..\data\raw\test_tabular_data\test_angles_dataset2.csv
   Shape: (62, 27)

📋 First 5 rows:


,frame_number,left_elbow_angle,right_elbow_angle,left_shoulder_angle,right_shoulder_angle,left_hip_angle,right_hip_angle,left_knee_angle,right_knee_angle,left_ankle_angle,...,right_shoulder_angle_velocity,left_hip_angle_velocity,right_hip_angle_velocity,left_knee_angle_velocity,right_knee_angle_velocity,left_ankle_angle_velocity,right_ankle_angle_velocity,shoulder_z_diff_velocity,hip_z_diff_velocity,torso_rotation_velocity
0,0,79.47,77.73,107.15,107.79,159.86,155.39,173.44,178.28,128.27,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,1,57.45,81.88,111.05,121.42,164.00,169.00,164.96,177.12,152.45,...,409.63,124.42,409.03,254.85,34.86,726.69,1696.51,0.32,0.20,0.13
2,2,81.89,58.79,116.24,116.44,176.34,173.71,176.52,178.63,146.12,...,149.67,370.86,141.55,347.42,45.38,190.24,1260.14,1.46,1.54,0.08
3,3,93.63,99.56,115.56,129.59,166.58,173.09,177.65,170.54,159.88,...,395.20,293.32,18.63,33.96,243.13,413.53,466.43,4.47,2.66,1.82
4,4,96.15,98.59,116.26,126.99,164.69,173.98,176.35,178.16,179.77,...,78.14,56.80,26.75,39.07,229.01,597.76,700.24,0.15,0.26,0.40


✅ No null values found.
✅ All angles within [0°, 180°]. No rows dropped.
📈 Angle columns statistics:


,left_elbow_angle,right_elbow_angle,left_shoulder_angle,right_shoulder_angle,left_hip_angle,right_hip_angle,left_knee_angle,right_knee_angle,left_ankle_angle,right_ankle_angle
count,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.00000,62.000000,62.000000,62.000000
mean,70.700323,73.916452,94.940161,101.582903,164.623548,166.386935,171.49629,172.552258,157.885645,154.562742
std,30.510880,31.135561,27.651322,30.665038,6.964392,8.118590,5.75090,5.657770,15.853244,16.896155
min,6.000000,0.710000,4.850000,2.130000,145.530000,148.120000,153.28000,160.870000,126.020000,117.910000
25%,47.812500,53.982500,77.692500,88.705000,161.470000,159.872500,168.92000,168.582500,145.692500,142.227500
50%,75.705000,76.405000,101.870000,107.140000,165.255000,167.215000,172.76000,174.270000,162.795000,155.025000
75%,98.080000,100.390000,118.585000,127.725000,168.002500,173.427500,176.30250,177.127500,169.600000,166.622500
max,111.250000,144.670000,124.840000,144.890000,179.990000,179.080000,179.89000,179.990000,179.980000,179.860000


In [8]:
# Load the scaler used during training
SCALER_PATH = Path("../data/models/scaler.pkl")
scaler = joblib.load(SCALER_PATH)

print("✅ Scaler loaded successfully.")
print(f"   Scaler path: {SCALER_PATH.resolve()}")

✅ Scaler loaded successfully.
   Scaler path: C:\Users\USER\Desktop\AI_PROJECT_2\data\models\scaler.pkl


In [9]:
metadata_cols = ["frame_number"]
feature_cols  = [col for col in df_raw.columns if col not in metadata_cols]

# --- Velocity noise cap (mirrors training preprocessing) ---
velocity_cols = [col for col in feature_cols if "velocity" in col]
df_raw[velocity_cols] = df_raw[velocity_cols].clip(upper=3000)
print(f"✅ Velocity values capped at 3000 deg/s ({len(velocity_cols)} columns).")

# --- Feature count guard ---
assert len(feature_cols) == scaler.n_features_in_, (
    f"Feature mismatch! Inference has {len(feature_cols)} features, "
    f"but scaler expects {scaler.n_features_in_}."
)

# --- Apply scaler ---
X_scaled = scaler.transform(df_raw[feature_cols])

df_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
df_scaled.insert(0, "frame_number", df_raw["frame_number"].values)

print("✅ Scaling complete.")
print(f"   Shape: {df_scaled.shape}")

✅ Velocity values capped at 3000 deg/s (13 columns).
✅ Scaling complete.
   Shape: (62, 27)


In [10]:
# Save scaled dataset for inference
OUTPUT_SCALED_PATH = Path("../data/processed/test_tabular_data/new_video_features_scaled2.csv")
OUTPUT_SCALED_PATH.parent.mkdir(parents=True, exist_ok=True)

df_scaled.to_csv(OUTPUT_SCALED_PATH, index=False)

print("✅ Scaled dataset saved.")
print(f"   Path: {OUTPUT_SCALED_PATH.resolve()}")

✅ Scaled dataset saved.
   Path: C:\Users\USER\Desktop\AI_PROJECT_2\data\processed\test_tabular_data\new_video_features_scaled2.csv


In [11]:
import joblib

# Load trained models
EXERCISE_SVM_MODEL_PATH    = Path("../data/models/svm_exercise_tuned.pkl")
CORRECTNESS_SVM_MODEL_PATH = Path("../data/models/svm_correctness_tuned.pkl")
EXERCISE_RF_MODEL_PATH        = Path("../data/models/rf_exercise_tuned.pkl")
CORRECTNESS_RF_MODEL_PATH     = Path("../data/models/rf_correctness_tuned.pkl")

exercise_model = joblib.load(EXERCISE_SVM_MODEL_PATH)
correctness_model = joblib.load(CORRECTNESS_SVM_MODEL_PATH)
exercise_rf_model = joblib.load(EXERCISE_RF_MODEL_PATH)
correctness_rf_model = joblib.load(CORRECTNESS_RF_MODEL_PATH)

print("✅ Models loaded successfully.")
print(f"   Exercise model   : {EXERCISE_SVM_MODEL_PATH.resolve()}")
print(f"   Correctness model: {CORRECTNESS_SVM_MODEL_PATH.resolve()}")
print(f"   Exercise RF model: {EXERCISE_RF_MODEL_PATH.resolve()}")
print(f"   Correctness RF model: {CORRECTNESS_RF_MODEL_PATH.resolve()}")

✅ Models loaded successfully.
   Exercise model   : C:\Users\USER\Desktop\AI_PROJECT_2\data\models\svm_exercise_tuned.pkl
   Correctness model: C:\Users\USER\Desktop\AI_PROJECT_2\data\models\svm_correctness_tuned.pkl
   Exercise RF model: C:\Users\USER\Desktop\AI_PROJECT_2\data\models\rf_exercise_tuned.pkl
   Correctness RF model: C:\Users\USER\Desktop\AI_PROJECT_2\data\models\rf_correctness_tuned.pkl


In [12]:
metadata_cols = ["frame_number"]
feature_cols  = [col for col in df_scaled.columns if col not in metadata_cols]

X = df_scaled[feature_cols]

# --- SVM Predictions ---
df_scaled["pred_exercise_svm"]    = exercise_model.predict(X)
df_scaled["pred_correctness_svm"] = correctness_model.predict(X)

# --- Random Forest Predictions ---
df_scaled["pred_exercise_rf"]    = exercise_rf_model.predict(X)
df_scaled["pred_correctness_rf"] = correctness_rf_model.predict(X)

print("✅ Predictions completed.")
display(df_scaled[[
    "frame_number",
    "pred_exercise_svm", "pred_correctness_svm",
    "pred_exercise_rf",  "pred_correctness_rf"
]].head(10))

✅ Predictions completed.


,frame_number,pred_exercise_svm,pred_correctness_svm,pred_exercise_rf,pred_correctness_rf
0,0,0,0,0,1
1,1,0,1,0,1
2,2,0,1,0,1
3,3,0,1,0,0
4,4,0,0,0,1
5,5,0,1,0,1
6,6,0,1,0,1
7,7,0,0,0,1
8,8,0,0,0,1
9,9,0,0,0,1


In [13]:
# --- Aggregate predictions for single video (majority vote) ---

def majority_vote(series):
    return series.value_counts().idxmax()

overall_predictions = {
    "exercise_svm": majority_vote(df_scaled["pred_exercise_svm"]),
    "correctness_svm": majority_vote(df_scaled["pred_correctness_svm"]),
    "exercise_rf": majority_vote(df_scaled["pred_exercise_rf"]),
    "correctness_rf": majority_vote(df_scaled["pred_correctness_rf"]),
}

print("✅ Overall prediction (entire video):")
for k, v in overall_predictions.items():
    print(f"   {k}: {v}")

✅ Overall prediction (entire video):
   exercise_svm: 0
   correctness_svm: 1
   exercise_rf: 0
   correctness_rf: 1
